# Lesson 4.1 - Azure Authentication and Context

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Load the Azure PowerShell module and inspect your context
- Capture subscription and tenant information for later scripts
- Prepare a safe resource-group deployment script

**Project file(s) produced:** `scripts/lesson-4.1-azure-auth-context.ps1` and `artifacts/azure-context.txt`

## Phase 4 Overview

This phase focuses on PowerShell in real DevOps workflows: Azure automation, provisioning patterns, and CI/CD integration. The goal is to turn local scripts into repeatable operational tools.

### Prerequisites
- Completion of Phase 3
- An Azure subscription if you want to run the cloud examples live
- A working PowerShell installation

## Concepts

Azure PowerShell is a set of modules that expose Azure management commands. Once you authenticate, PowerShell keeps a **context** that records the current subscription and tenant so later commands know where to operate.

A safe automation script often writes a local plan first and only performs the cloud action when the inputs are confirmed.

Reference: [Azure PowerShell overview](https://learn.microsoft.com/en-us/powershell/azure/overview)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap

lesson_dir = Path('study-plan/topic-powershell/phase-4-cloud-devops')
artifact_dir = lesson_dir / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

def run_powershell(script_text: str, script_name: str) -> int:
    script_path = lesson_dir / script_name
    script_path.write_text(script_text, encoding='utf-8')
    executable = shutil.which('pwsh') or shutil.which('powershell')
    if executable:
        result = subprocess.run([executable, '-NoLogo', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(script_path)], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        return result.returncode
    print(f'{script_path.name} written. Run it in PowerShell to see the live output.')
    return 0

## Code-Along: Step 1 - Inspect Azure context

This script is safe to run even when you are not signed in. It checks whether the Az modules and context are available, then writes a small local summary.

In [ ]:
script = textwrap.dedent(r'''
$reportPath = Join-Path $PSScriptRoot 'artifacts/azure-context.txt'
$lines = @()
$lines += 'Azure PowerShell context check'
$lines += '============================='
$lines += "Az module available: $([bool](Get-Module Az.Accounts -ListAvailable))"
$context = Get-AzContext -ErrorAction SilentlyContinue
if ($context) {
    $lines += "Subscription: $($context.Subscription.Name)"
    $lines += "Tenant: $($context.Tenant.Id)"
} else {
    $lines += 'No Azure context is loaded yet.'
}
$lines | Set-Content -Path $reportPath
Get-Content $reportPath
''')
run_powershell(script, 'lesson-4.1-azure-context.ps1')

## Code-Along: Step 2 - Draft a safe deployment script

This step writes a script that shows how you would provision a resource group in a repeatable way. The important habit is to make the inputs explicit and keep the deployment plan easy to inspect.

In [ ]:
script = textwrap.dedent(r'''
$resourceGroupName = 'rg-powershell-study-plan'
$location = 'eastus'
$tags = @{ Project = 'PowerShell Study Plan'; Environment = 'dev' }

# Uncomment after authenticating with Connect-AzAccount
# New-AzResourceGroup -Name $resourceGroupName -Location $location -Tag $tags -WhatIf

[pscustomobject]@{ ResourceGroup = $resourceGroupName; Location = $location; TagCount = $tags.Count } | Format-List
''')
run_powershell(script, 'lesson-4.1-resource-group-plan.ps1')

## Exercise

Change the planned resource group name and add one more tag that identifies your team or lab.

**Your task:**
1. Update the resource group name.
2. Add one more tag.
3. Confirm the summary still prints cleanly.

**Expected result:** a safe deployment script template you can reuse in later lessons.

In [ ]:
# TODO: change the resource group naming and add a team tag
exercise_script = textwrap.dedent(r'''
Write-Host "TODO: customize the Azure plan"
''')
run_powershell(exercise_script, 'lesson-4.1-exercise.ps1')

## Key Takeaways

- Azure PowerShell adds cloud management commands to the same scripting language you already use locally.
- Context matters, because it tells Azure which subscription and tenant to target.
- A local plan file is safer than jumping straight into a production change.

---

**Next:** [Lesson 4.2 - Resource Provisioning and Automation](./lesson-4.2-resource-provisioning-automation.ipynb)